In [2]:
import pandas as pd
from scipy import stats

from sqlalchemy import text, create_engine, URL

username = "root"
password = "Kuber@123"
server = "localhost"
port = 3306
database = "Car2Insights"

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=username,
    password=password,
    host=server,
    port=port,
    database=database
)

engine = create_engine(connection_url)
# Load order-level category spending
query = """
SELECT
    COALESCE(p.product_category_name, 'Unknown') AS product_category_name,
    oi.order_id,
    SUM(oi.price + oi.freight_value) AS order_value
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    COALESCE(p.product_category_name, 'Unknown'),
    oi.order_id
"""

anova_df = pd.read_sql(query, engine)

# Keep categories with at least 30 observations
category_counts = anova_df["product_category_name"].value_counts()
valid_categories = category_counts[category_counts >= 30].index

anova_filtered = anova_df[
    anova_df["product_category_name"].isin(valid_categories)
]

# Create groups
groups = [
    group["order_value"].dropna().values
    for _, group in anova_filtered.groupby("product_category_name")
]

# One-Way ANOVA
f_statistic, p_value = stats.f_oneway(*groups)

print("Number of categories:", len(groups))
print("F-statistic:", f_statistic)
print("P-value:", p_value)

# Decision
alpha = 0.05

if p_value < alpha:
    print("\nDecision: Reject H0")
    print("Conclusion: Average order value differs significantly across product categories.")
else:
    print("\nDecision: Fail to Reject H0")
    print("Conclusion: No statistically significant difference in average order value across product categories.")

Number of categories: 63
F-statistic: 175.56726052405648
P-value: 0.0

Decision: Reject H0
Conclusion: Average order value differs significantly across product categories.
